# backwardCGM-PD — Baseline pdglasso trên dữ liệu mô phỏngThực nghiệm **mở rộng** do nhóm bổ sung, không có trong bài báo gốc.Phần mô phỏng của bài báo chỉ đối chiếu `tau` với `submodel`, mà cả hai đều làthủ tục từng bước thuộc phương pháp đề xuất — nghĩa là không có đối thủ nào nằmngoài phương pháp. Notebook này chạy hai baseline pdglasso (trên ma trận hiệpphương sai và trên ma trận tương quan) lên đúng 320 tập dữ liệu mô phỏng đã dùngở tám notebook `kaggle-simulation-*`, chấm bằng cùng sáu độ đo phục hồi.Hãy **Add Input** dataset `backwardCGM-PD`.

In [ ]:
import importlib.util, subprocess, sys
required = {"rdata": "rdata>=0.11", "networkx": "networkx>=3.0", "joblib": "joblib>=1.3"}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies: OK")

In [ ]:
from pathlib import Path
import json, shutil, zipfile
import pandas as pd
from IPython.display import Image, display

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/backwardCGM-PD")
RESULTS = Path("/kaggle/working/baseline-simulation-results")
RESULTS.mkdir(parents=True, exist_ok=True)

archives = list(INPUT_ROOT.rglob("backwardCGM-PD-kaggle-dataset.zip"))
scripts = list(INPUT_ROOT.rglob("python-port/experiments/simulation.py"))
if archives:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(WORK_ROOT)
elif scripts:
    shutil.copytree(scripts[0].parents[2], WORK_ROOT, dirs_exist_ok=True)
else:
    raise FileNotFoundError("Hãy Add Input dataset backwardCGM-PD")

PORT_ROOT = WORK_ROOT / "python-port"
data_files = [
    WORK_ROOT / f"simulation/simulated-data/simdf_{s}_{p}.RData"
    for s in ("11", "22") for p in (8, 12, 16, 20)
]
missing_files = [str(p) for p in data_files if not p.exists()]
if missing_files:
    raise FileNotFoundError("Dataset Simulation không đầy đủ: " + str(missing_files))
print("Dataset: OK\nOutput:", RESULTS)

In [ ]:
# Hai thực nghiệm mở rộng do nhóm viết thêm, chưa có trong dataset gốc nên được
# nhúng thẳng vào notebook. Mã dưới đây giống hệt tệp cùng tên trong kho
# github.com/nhantrnh/DataMining.
SCRIPT = PORT_ROOT / "experiments" / "baseline_simulation.py"
SCRIPT.write_text('"""Baseline comparison on the simulated data of Roverato & Nguyen (2024).\n\nThe article evaluates two search strategies on the twin lattice (``tau``) and\non the model inclusion lattice (``submodel``).  Both belong to the proposed\nmethod, so the simulation study of the paper contains no external competitor.\n\nThis script adds two baselines that are standard in the colored graphical\nmodel literature and that the article itself uses for the air quality data:\n\n``pdglasso-cov``\n    pdRCON graphical lasso applied to the sample covariance matrix.\n\n``pdglasso-cor``\n    the same estimator applied to the sample correlation matrix, i.e. after\n    standardising every variable.  The article stresses that pdglasso is not\n    scale invariant, so the two variants are genuinely different baselines.\n\nBoth are scored with the recovery metrics of Table 3.2, exactly the ones used\nfor the stepwise procedures, which makes the comparison directly readable\nagainst the numbers reproduced in Chapter 4 of the report.\n\nUsage::\n\n    python experiments/baseline_simulation.py --scenario A --p 8 \\\n        --replicates 20 --output results/baseline-A-p8.json\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nREPO_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(REPO_ROOT / "src"))\n\nfrom backward_cgm_pd.article_graphs import article_scenario_graph\nfrom backward_cgm_pd.io import load_simulated_datasets, write_json\nfrom backward_cgm_pd.metrics import average_metrics, recovery_metrics\nfrom backward_cgm_pd.pdglasso import select_pdglasso\n\nSCENARIO_CODE = {"A": "11", "B": "22"}\n\n\ndef saved_data_path(scenario: str, p: str) -> Path:\n    return (\n        REPO_ROOT\n        / "data"\n        / "simulated-data"\n        / f"simdf_{SCENARIO_CODE[scenario]}_{p}.RData"\n    )\n\n\ndef correlation_from_data(data: np.ndarray) -> np.ndarray:\n    """Sample correlation matrix, i.e. the covariance of standardised data."""\n    return np.corrcoef(data, rowvar=False)\n\n\ndef run_baseline(\n    scenario: str,\n    p: str,\n    *,\n    replicates: int,\n    points: int,\n    gamma_ebic: float,\n    verbose: bool,\n) -> dict[str, object]:\n    datasets = load_simulated_datasets(saved_data_path(scenario, p))[:replicates]\n    truth = article_scenario_graph(scenario, int(p))\n\n    rows: list[dict[str, object]] = []\n    for index, data in enumerate(datasets, 1):\n        n = data.shape[0]\n        for variant, matrix in (\n            ("pdglasso-cov", np.cov(data, rowvar=False, ddof=1)),\n            ("pdglasso-cor", correlation_from_data(data)),\n        ):\n            started = time.perf_counter()\n            selection = select_pdglasso(\n                matrix, n, points=points, gamma_ebic=gamma_ebic\n            )\n            runtime = time.perf_counter() - started\n            metrics = recovery_metrics(selection.graph, truth)\n            rows.append(\n                {\n                    "replicate": index,\n                    "method": variant,\n                    "runtime_seconds": runtime,\n                    "lambda1": selection.best_lambdas[0],\n                    "lambda2": selection.best_lambdas[1],\n                    "number_edges": len(selection.graph.E),\n                    "number_parameters": selection.graph.n_parameters,\n                    "model": selection.graph,\n                    "metrics": metrics.to_dict(),\n                }\n            )\n            if verbose:\n                print(\n                    f"[{scenario}/p={p}] replicate {index}/{len(datasets)} "\n                    f"{variant}: {runtime:.2f}s |E|={len(selection.graph.E)}",\n                    flush=True,\n                )\n\n    summary: list[dict[str, object]] = []\n    for variant in ("pdglasso-cov", "pdglasso-cor"):\n        subset = [row for row in rows if row["method"] == variant]\n        runtimes = [float(row["runtime_seconds"]) for row in subset]\n        edges = [int(row["number_edges"]) for row in subset]\n        averaged = average_metrics(\n            [recovery_metrics(row["model"], truth) for row in subset]\n        )\n        summary.append(\n            {\n                "scenario": scenario,\n                "p": int(p),\n                "method": variant,\n                "replicates": len(subset),\n                "mean_runtime": float(np.mean(runtimes)),\n                "sd_runtime": float(np.std(runtimes, ddof=1)) if len(runtimes) > 1 else 0.0,\n                "mean_number_edges": float(np.mean(edges)),\n                **averaged,\n            }\n        )\n\n    return {\n        "settings": {\n            "scenario": scenario,\n            "p": int(p),\n            "replicates": len(datasets),\n            "points": points,\n            "gamma_ebic": gamma_ebic,\n            "source": "saved",\n        },\n        "truth": truth,\n        "runs": rows,\n        "summary": summary,\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--scenario", choices=["A", "B"], required=True)\n    parser.add_argument("--p", choices=["8", "12", "16", "20"], required=True)\n    parser.add_argument("--replicates", type=int, default=20)\n    parser.add_argument("--points", type=int, default=10)\n    parser.add_argument("--gamma-ebic", type=float, default=0.0)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--verbose", action="store_true")\n    args = parser.parse_args()\n\n    result = run_baseline(\n        args.scenario,\n        args.p,\n        replicates=args.replicates,\n        points=args.points,\n        gamma_ebic=args.gamma_ebic,\n        verbose=args.verbose,\n    )\n\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    write_json(result, args.output)\n    summary_path = args.output.with_name(f"{args.output.stem}-summary.csv")\n    pd.DataFrame(result["summary"]).to_csv(summary_path, index=False)\n    print(f"Wrote {args.output}")\n    print(f"Wrote {summary_path}")\n    print(pd.DataFrame(result["summary"]).to_string(index=False))\n\n\nif __name__ == "__main__":\n    main()\n')

# Script định vị dữ liệu qua REPO_ROOT/data/simulated-data, với REPO_ROOT là
# thư mục python-port; dataset Kaggle đặt chúng ở simulation/simulated-data nên
# tạo một liên kết cho khớp.
data_dir = PORT_ROOT / "data"
data_dir.mkdir(exist_ok=True)
target = data_dir / "simulated-data"
if not target.exists():
    target.symlink_to(WORK_ROOT / "simulation/simulated-data")
assert (target / "simdf_11_8.RData").exists(), "Liên kết dữ liệu không đúng"
print("Đã ghi", SCRIPT)

## Chạy tám cấu hìnhHai kịch bản A và B, bốn giá trị `p`, mỗi cấu hình 20 replicate. pdglasso rẻ hơnnhiều so với tìm kiếm từng bước nên toàn bộ chạy trong ít phút.

In [ ]:
import subprocess, sys, time

started = time.perf_counter()
for scenario in ("A", "B"):
    for p in ("8", "12", "16", "20"):
        output = RESULTS / f"baseline-{scenario}-p{p}.json"
        if output.exists():
            print(f"Bỏ qua {scenario}/p={p}: đã có kết quả")
            continue
        command = [
            sys.executable, "-u", str(PORT_ROOT / "experiments/baseline_simulation.py"),
            "--scenario", scenario, "--p", p, "--replicates", "20",
            "--output", str(output),
        ]
        print("Running:", " ".join(command), flush=True)
        subprocess.run(command, cwd=PORT_ROOT, check=True)
print(f"\nTổng thời gian: {time.perf_counter() - started:.1f}s")

## Tổng hợp kết quả

In [ ]:
import glob
frames = [pd.read_csv(f) for f in sorted(glob.glob(str(RESULTS / "*-summary.csv")))]
summary = pd.concat(frames, ignore_index=True).sort_values(["scenario", "p", "method"])
summary.to_csv(RESULTS / "baseline-all-configurations.csv", index=False)
display(summary)

archive = shutil.make_archive("/kaggle/working/baseline-simulation-results", "zip",
                              root_dir=RESULTS)
print("Download:", archive)

**Đọc kết quả:** so sánh cột `ePPV` và `eTPR` của hai dòng `pdglasso-*` với kếtquả `tau`/`submodel` trong các notebook `kaggle-simulation-*` cùng cấu hình.Giá trị `sPPV` bị khuyết ở một số cấu hình kịch bản B là đúng hành vi của mã Rgốc: khi một replicate không chọn được phát biểu đối xứng nào thì mẫu số bằngkhông, và trung bình được tính không loại giá trị khuyết.